In [1]:
import cv2
import json
import os
import torch
from ultralytics.models import YOLO

# --- CONFIG ---
VIDEO_FOLDER = "D:\\GitHub\\BaseballPitch\\data\\fuller_finetuning_dataset\\videos"
OUTPUT_BASE = "D:\\GitHub\\BaseballPitch\\data\\fuller_finetuning_dataset\\labels"
SPLIT = "val"  # 'train' or 'val' or 'test'
PITCH_TYPE = "ST - Sweeper"  # Set the pitch type for labeling
POSE_MODEL = 'D:\\GitHub\\BaseballPitch\\modules\\pitcher_detection\\runs\\best.pt'  # Detect people with pose keypoints
CONF = 0.25  # Confidence threshold for pose detection
IMG_SIZE = 1280
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
# --------------

In [10]:
def label_pitch_type(videos, type):
    
    """ Loop through videos and label them """
    
    print("Starting labeling...\n")
    for i, video in enumerate(videos, 1):
        print(f"Processing video: {video}")
        print(f"[Video {i}/{len(videos)}]")

        # Define the text file path
        txt_path = f"{OUTPUT_BASE}/{SPLIT}/{type}/{video.replace('.mp4', '.txt')}"
        
        # Open in 'write' mode to clear any old data and create a fresh file
        open(txt_path, 'w').close()

        # Access the results (stream=True added to fix the RAM warning)
        results = model(video_dir + "/" + video,
                        conf=CONF, 
                        imgsz=IMG_SIZE, 
                        device=DEVICE, 
                        verbose=False,
                        stream=True) 
        
        # Get labels from results frame by frame
        for result in results:
            if len(result.boxes) > 0:
                # Finds the person it is most confident about
                best_idx = result.boxes.conf.argmax()
                top_person_result = result[best_idx]
                
                # Ultralytics natively appends the detection AND a newline
                top_person_result.save_txt(txt_path)  
            else:
                # If no pitcher is found, append an empty line to keep frames synced
                with open(txt_path, 'a') as f:
                    f.write('\n')
                    
        print(f"Results obtained for {video}")

# Load a custom model
model = YOLO(POSE_MODEL)

SPLIT = "train"  # 'train' or 'val' or 'test'
pitch_types = ["FF - Fastball", "SL - Slider", "SI - Sinker", "CH - Changeup", "ST - Sweeper"]

for pitch_type in pitch_types:
    # Get list of videos to label
    video_dir = f"{VIDEO_FOLDER}/{SPLIT}/{pitch_type}"
    
    # Safety check in case a directory doesn't exist
    if not os.path.exists(video_dir):
        continue
        
    videos = sorted([f for f in os.listdir(video_dir) if f.endswith(".mp4")])

    # Check if there are videos to label
    if not videos:
        print(f"No videos found in {video_dir}")
        continue
        
    # Label the videos
    label_pitch_type(videos, pitch_type)

Starting labeling...

Processing video: PitchType-FF_Zone-11_PlayID-008f8915-0770-3e3e-99bf-70d08a4d8609_Date-2025-05-24.mp4
[Video 1/2156]
Results obtained for PitchType-FF_Zone-11_PlayID-008f8915-0770-3e3e-99bf-70d08a4d8609_Date-2025-05-24.mp4
Processing video: PitchType-FF_Zone-11_PlayID-00cde179-4638-4a85-b76d-325ecee43955_Date-2023-06-27.mp4
[Video 2/2156]
Results obtained for PitchType-FF_Zone-11_PlayID-00cde179-4638-4a85-b76d-325ecee43955_Date-2023-06-27.mp4
Processing video: PitchType-FF_Zone-11_PlayID-0127f85b-8731-3d29-90d3-9b33bd63c8ee_Date-2025-06-23.mp4
[Video 3/2156]
Results obtained for PitchType-FF_Zone-11_PlayID-0127f85b-8731-3d29-90d3-9b33bd63c8ee_Date-2025-06-23.mp4
Processing video: PitchType-FF_Zone-11_PlayID-019be2e5-1a60-4689-9dc6-d883f876e863_Date-2024-09-22.mp4
[Video 4/2156]
Results obtained for PitchType-FF_Zone-11_PlayID-019be2e5-1a60-4689-9dc6-d883f876e863_Date-2024-09-22.mp4
Processing video: PitchType-FF_Zone-11_PlayID-04a0905c-b3e4-44fc-9435-5ea9b3ac19bd

In [2]:
# Load a custom model
model = YOLO("D:/GitHub/BaseballPitch/modules/pitcher_detection/runs/best.pt")

# Capture video from the camera
cap = cv2.VideoCapture("D:/GitHub/BaseballPitch/data/fuller_finetuning_dataset/videos/train/FF - Fastball/PitchType-FF_Zone-11_PlayID-00cde179-4638-4a85-b76d-325ecee43955_Date-2023-06-27.mp4")

# Loop through the video frames
while cap.isOpened():
    # Read a frame from the video
    success, frame = cap.read()

    if success:
        # Run YOLO inference on the frame
        results = model(frame, device=DEVICE, verbose = False)

        # Visualize the results on the frame
        annotated_frame = results[0].plot()

        # Display the annotated frame
        cv2.imshow("YOLO Inference", annotated_frame)

        # Break the loop if 'q' is pressed
        if cv2.waitKey(1) & 0xFF == ord("q"):
            break
    else:
        # Break the loop if the end of the video is reached
        break

# Release the video capture object and close the display window
cap.release()
cv2.destroyAllWindows()

Surprisingly, my own video displayer works faster than the Ultralytics one (which I didn't find till later); I doubt this is just because of me skipping the edge connection drawings.

This makes me worry about having found their label saver, and if mine would end up faster? But hey, saves me the time to have to figure out how to write it or convert formats!

In [9]:
# Load a custom model
model = YOLO("D:/GitHub/BaseballPitch/modules/pitcher_detection/runs/best.pt")

# Capture video from the camera
cap = cv2.VideoCapture("D:/GitHub/BaseballPitch/data/curated_finetuning_dataset/videos/val/CH - Changeup/PitchType-CH_Zone-13_PlayID-0c9e10ef-a750-33be-980b-4d1573c7df45_Date-2025-07-25.mp4")

# Loop through the video frames
while True:
    ret, frame = cap.read()
    if not ret:
        break
    
    detected_boxes = []
    detected_keypoints = []
    
    # Run inference
    results = model(frame.copy(), conf=CONF, imgsz=IMG_SIZE, device=DEVICE, verbose=False)
    result = results[0]
    if result.boxes is not None and len(result.boxes) > 0:
        boxes = result.boxes.xyxy
        if isinstance(boxes, torch.Tensor):
            boxes = boxes.cpu().numpy()
        kpts = result.keypoints.data
        if isinstance(kpts, torch.Tensor):
            kpts_data = kpts.cpu().numpy()

        for i, box in enumerate(boxes):
            x1, y1, x2, y2 = map(int, box[:4])
            detected_boxes.append((x1, y1, x2, y2))
            detected_keypoints.append(kpts[i])

    for i, (bx1, by1, bx2, by2) in enumerate(detected_boxes):
        color = (0, 255, 255)
        cv2.rectangle(frame, (bx1, by1), (bx2, by2), color, 2)
        cv2.putText(frame, f"Pitcher {i+1}", (bx1, by1 - 10),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 2)
        if i < len(detected_keypoints):
            for kx, ky, kconf in detected_keypoints[i]:
                if kconf > 0.5:
                    cv2.circle(frame, (int(kx), int(ky)), 3, (0, 0, 255), -1)

    # Display the results
    cv2.imshow('Pitch Inference', frame)

    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()


Here is one for writing the videos with the annotations as well.

In [12]:
# Load a custom model
model = YOLO("D:/GitHub/BaseballPitch/modules/pitcher_detection/runs/best.pt")

# Capture video from the camera
cap = cv2.VideoCapture("D:/GitHub/BaseballPitch/data/curated_finetuning_dataset/videos/val/CH - Changeup/PitchType-CH_Zone-13_PlayID-0c9e10ef-a750-33be-980b-4d1573c7df45_Date-2025-07-25.mp4")
out = cv2.VideoWriter('output.mp4', -1, 59.0, (1280,720))

# Loop through the video frames
while True:
    ret, frame = cap.read()
    if not ret:
        break
    
    detected_boxes = []
    detected_keypoints = []
    
    # Run inference
    results = model(frame.copy(), conf=CONF, imgsz=IMG_SIZE, device=DEVICE, verbose=False)
    result = results[0]
    if result.boxes is not None and len(result.boxes) > 0:
        boxes = result.boxes.xyxy
        if isinstance(boxes, torch.Tensor):
            boxes = boxes.cpu().numpy()
        kpts = result.keypoints.data
        if isinstance(kpts, torch.Tensor):
            kpts_data = kpts.cpu().numpy()

        for i, box in enumerate(boxes):
            x1, y1, x2, y2 = map(int, box[:4])
            detected_boxes.append((x1, y1, x2, y2))
            detected_keypoints.append(kpts[i])

    for i, (bx1, by1, bx2, by2) in enumerate(detected_boxes):
        color = (0, 255, 255)
        cv2.rectangle(frame, (bx1, by1), (bx2, by2), color, 2)
        cv2.putText(frame, f"Pitcher {i+1}", (bx1, by1 - 10),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 2)
        if i < len(detected_keypoints):
            for kx, ky, kconf in detected_keypoints[i]:
                if kconf > 0.5:
                    cv2.circle(frame, (int(kx), int(ky)), 3, (0, 0, 255), -1)

    # Display the results
    cv2.imshow('Pitch Inference', frame)

    # Write the frame to the output video
    out.write(frame)
    
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
out.release()
cv2.destroyAllWindows()


In [2]:
# Load a custom model
model = YOLO("D:/GitHub/BaseballPitch/modules/pitcher_detection/runs/best.pt")

# Access the results
results = model("D:/GitHub/BaseballPitch/data/curated_finetuning_dataset/videos/val/CH - Changeup/PitchType-CH_Zone-13_PlayID-0c9e10ef-a750-33be-980b-4d1573c7df45_Date-2025-07-25.mp4",
                conf=CONF, imgsz=IMG_SIZE, device=DEVICE, verbose=False)

for result in results:
    result.save_txt("output.txt")  # Save results to a text file
    xy = result.keypoints.xy  # x and y coordinates
    xyn = result.keypoints.xyn  # normalized
    kpts = result.keypoints.data  # x, y, visibility (if available)

WARNING 
Inference results will accumulate in RAM unless `stream=True` is passed, which can cause out-of-memory errors for large
sources or long-running streams and videos. See https://docs.ultralytics.com/modes/predict/ for help.

Example:
    results = model(source=..., stream=True)  # generator of Results objects
    for r in results:
        boxes = r.boxes  # Boxes object for bbox outputs
        masks = r.masks  # Masks object for segment masks outputs
        probs = r.probs  # Class probabilities for classification outputs

